# 01 — Data Preprocessing (Benign vs Malignant)

Preprocessing pipeline for liver CT images used in **Stage 2** (benign vs malignant classification).

## Pipeline Steps
1. **Load** grayscale CT image
2. **Denoise** with Gaussian blur
3. **Segment** body region using Otsu thresholding
4. **Extract ROI** via largest contour + bounding box with padding
5. **Morphological closing** to fill small holes in the mask
6. **Resize** to 224×224 (bicubic interpolation)
7. **Normalize** to ImageNet mean/std
8. **Augment** benign class to balance with malignant (target: 317 per class)

> **Note:** This notebook processes raw images from `data/train/` and overwrites them in place.  
> Since data is already preprocessed, **re-running is not required**.

## Configuration

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path

# --- Paths ---
BENIGN_DIR    = "../data/train/benign"
MALIGNANT_DIR = "../data/train/malignant"

# --- Preprocessing ---
IMG_SIZE       = 224
IMAGENET_MEAN  = np.array([0.485, 0.456, 0.406])
IMAGENET_STD   = np.array([0.229, 0.224, 0.225])
BLUR_KERNEL    = (5, 5)
CONTOUR_PAD    = 5         # padding around bounding box (px)
MORPH_KERNEL   = np.ones((3, 3), np.uint8)

# --- Augmentation ---
TARGET_PER_CLASS = 317     # target count for class balance
ROTATION_RANGE   = 25      # degrees
BRIGHTNESS_RANGE = (0.7, 1.3)
SHIFT_RANGE      = 0.2     # fraction of total dimension
ZOOM_RANGE       = 0.3
SHEAR_RANGE      = 10      # degrees
IMAGE_EXTS       = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

## Preprocessing Functions

In [ ]:
def process_image(img_path: str) -> np.ndarray | None:
    """
    Load a grayscale CT image, extract ROI, resize, and normalize.
    
    Returns:
        Normalized RGB image (H, W, 3) with ImageNet mean/std,
        or None if the image cannot be read.
    """
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"[WARN] Cannot read image: {img_path}")
        return None

    # Handle invalid pixel values
    img = np.nan_to_num(img, nan=0, posinf=255, neginf=0).astype(np.uint8)

    # Denoise
    blurred = cv2.GaussianBlur(img, BLUR_KERNEL, 0)

    # Otsu thresholding to segment body region
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Find contours and extract largest (assumed to be body)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if not contours:
        roi = img
    else:
        largest = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(largest)

        # Create mask from contour + morphological closing
        mask = np.zeros_like(img)
        cv2.drawContours(mask, [largest], -1, 255, -1)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, MORPH_KERNEL, iterations=1)
        masked = cv2.bitwise_and(img, img, mask=mask)

        # Crop with padding
        y0 = max(0, y - CONTOUR_PAD)
        y1 = min(img.shape[0], y + h + CONTOUR_PAD)
        x0 = max(0, x - CONTOUR_PAD)
        x1 = min(img.shape[1], x + w + CONTOUR_PAD)
        roi = masked[y0:y1, x0:x1]

        if roi.size == 0:
            roi = img

    # Resize to target size
    roi = cv2.resize(roi, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_CUBIC)

    # Convert grayscale to RGB and normalize with ImageNet stats
    rgb = cv2.cvtColor(roi, cv2.COLOR_GRAY2RGB)
    normalized = (rgb.astype(np.float32) / 255.0 - IMAGENET_MEAN) / IMAGENET_STD

    return normalized


def denormalize_to_uint8(img: np.ndarray) -> np.ndarray:
    """Convert ImageNet-normalized image back to uint8 RGB for saving."""
    rgb = ((img * IMAGENET_STD + IMAGENET_MEAN) * 255.0).clip(0, 255).astype(np.uint8)
    return rgb

## Augmentation Functions

Data augmentation using OpenCV to balance the benign class up to the target count.  
Augmentations applied: random rotation, horizontal flip, brightness adjustment.

In [ ]:
def augment_image(img: np.ndarray) -> np.ndarray:
    """
    Apply random augmentation to a uint8 RGB image.
    Augmentations: rotation, horizontal flip, brightness shift.
    """
    h, w = img.shape[:2]

    # Random rotation
    angle = np.random.uniform(-ROTATION_RANGE, ROTATION_RANGE)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101)

    # Random horizontal flip
    if np.random.random() > 0.5:
        img = cv2.flip(img, 1)

    # Random brightness
    factor = np.random.uniform(*BRIGHTNESS_RANGE)
    img = np.clip(img.astype(np.float32) * factor, 0, 255).astype(np.uint8)

    return img


def list_images(folder: str) -> list[str]:
    """List all image files in a directory."""
    return sorted([f for f in os.listdir(folder) if f.lower().endswith(IMAGE_EXTS)])

## Run Preprocessing

In [ ]:
def process_directory(input_dir: str, output_dir: str, target_count: int):
    """
    Process all images in input_dir and save to output_dir.
    If the class has fewer images than target_count, augment to fill the gap.
    """
    Path(output_dir).mkdir(parents=True, exist_ok=True)
    images = list_images(input_dir)
    print(f"[INFO] Processing: {input_dir}")
    print(f"[INFO] Found {len(images)} images")

    # Step 1: Preprocess and save each image
    processed = 0
    for filename in images:
        img_path = os.path.join(input_dir, filename)
        result = process_image(img_path)
        if result is None:
            continue

        out_path = os.path.join(output_dir, filename)
        rgb_uint8 = denormalize_to_uint8(result)
        cv2.imwrite(out_path, cv2.cvtColor(rgb_uint8, cv2.COLOR_RGB2BGR))
        processed += 1

    print(f"[INFO] Preprocessed {processed}/{len(images)} images")

    # Step 2: Augment if needed for class balance
    current = len(list_images(output_dir))
    need = max(0, target_count - current)

    if need == 0:
        print(f"[INFO] Class already balanced ({current} images). Skipping augmentation.")
        return

    print(f"[INFO] Need {need} augmented images to reach target ({target_count})")
    source_files = list_images(output_dir)
    created = 0
    idx = 0

    while created < need:
        src_name = source_files[idx % len(source_files)]
        src_path = os.path.join(output_dir, src_name)
        src_img = cv2.imread(src_path, cv2.IMREAD_COLOR)
        if src_img is None:
            idx += 1
            continue

        src_rgb = cv2.cvtColor(src_img, cv2.COLOR_BGR2RGB)
        aug_rgb = augment_image(src_rgb)
        aug_bgr = cv2.cvtColor(aug_rgb, cv2.COLOR_RGB2BGR)

        aug_name = f"aug_{created:04d}_{src_name}"
        cv2.imwrite(os.path.join(output_dir, aug_name), aug_bgr)
        created += 1
        idx += 1

        if created % 20 == 0 or created == need:
            print(f"[INFO] Augmentation progress: {created}/{need}")

    print(f"[DONE] Final count: {len(list_images(output_dir))} images in {output_dir}")

## Execute

Process benign images (with augmentation for class balance) and malignant images.

In [ ]:
# Process benign (with augmentation to reach target)
print("=" * 60)
print("Processing BENIGN class")
print("=" * 60)
process_directory(BENIGN_DIR, BENIGN_DIR, target_count=TARGET_PER_CLASS)

print()

# Process malignant (no augmentation needed)
print("=" * 60)
print("Processing MALIGNANT class")
print("=" * 60)
process_directory(MALIGNANT_DIR, MALIGNANT_DIR, target_count=TARGET_PER_CLASS)

print()
print("=" * 60)
print("Preprocessing complete.")
print(f"  Benign:    {len(list_images(BENIGN_DIR))} images")
print(f"  Malignant: {len(list_images(MALIGNANT_DIR))} images")
print("=" * 60)